# 03.2 — Machine Learning Direct Multi-Horizon Benchmark (Validation Protocol v1.0.21)

Đánh giá chuẩn hóa 16 kịch bản học máy dạng bảng đa bước trực tiếp (**SC-12 đến SC-27**), bao gồm:
- **4 Họ Mô hình**: Linear Regression (`lr_direct`), Random Forest (`rf_direct`), XGBoost (`xgboost_direct`), LightGBM (`lightgbm_direct`).
- **4 Cấp độ Đặc trưng Bảng ($F_0, F_1, F_2, F_3$)**:
  * **$F_0$ (36 biến)**: Khí tượng bề mặt tức thời + 4 thành phần gió điều hòa (không có target lag).
  * **$F_1$ (53 biến)**: $F_0$ + `rainfall_lag_0` + 16 biến trễ (lags 1, 2, 3, 7 của 4 chuỗi khí quyển).
  * **$F_2$ (65 biến)**: $F_1$ + 12 biến thống kê cuộn nhân quả (Mean & Std trên cửa sổ 7, 14, 30 ngày).
  * **$F_3$ (70 biến)**: $F_2$ + 5 đặc trưng lịch biểu và chu kỳ mùa vụ điều hòa.

### Hai Mục tiêu Thực nghiệm Cốt lõi:
1. **Benchmark A (Scientific Feature Construction Ablation)**: Khảo sát đơn biến trên LightGBM Direct qua $F_0 \to F_3$ để chọn tập đặc trưng tối ưu $F^*$ trên 724 Validation origins.
2. **Benchmark B (Model Family Comparison)**: So sánh hiệu năng tương đối giữa 4 họ mô hình Machine Learning trên toàn bộ 16 kịch bản với Single-Fit trên 6,293 mẫu Dev Train chung.

In [1]:
import sys
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

from src.evaluation.protocols import (
    load_canonical_data,
    build_purged_direct_samples,
    compute_benchmark_metrics,
    CANONICAL_DEV_TRAIN_MASE_SCALE,
    ExecutionStatus,
)
from src.evaluation.metrics import evaluate_point_forecast_metrics
from src.featurengineering.builder import FeatureBuilder, FeatureConfig

# Nạp dữ liệu chuẩn hóa (Dev Train và Validation, loại bỏ hoàn toàn Test)
train_full, val_set, _ = load_canonical_data()
print(f"Loaded canonical data: Dev Train = {len(train_full):,} ngày, Validation = {len(val_set):,} ngày")


## 2. Xây dựng 4 Bộ Đặc trưng Chuẩn tắc ($F_0, F_1, F_2, F_3$)

Tuân thủ nghiêm ngặt **Quy tắc Phân lập Đặc trưng**:
- Học tham số imputation `_imputation_stats` thuần túy trên Dev Train.
- Trích xuất tập Train với common origin mask (loại 29 dòng warmup đầu tiên) để cố định chính xác **6,293 mẫu huấn luyện** cho mọi cấp độ.
- Áp dụng **Mode B** (`context_df=train_full`) trên Validation để bảo toàn trọn vẹn **724 Validation origins** mà không tạo ra bất kỳ giá trị NaN nào.

In [2]:
num_val_origins = 724
df_eval = pd.concat([train_full, val_set], ignore_index=True)
y_eval = df_eval['Lượng mưa'].astype(float).values
dev_train_len = len(train_full)  # 6329
origin_indices = [dev_train_len + i for i in range(num_val_origins)]
origin_dates = val_set['Ngày'].iloc[:num_val_origins].values

# Ground truth target window (724, 7)
y_val_true = np.array([y_eval[idx + 1 : idx + 8] for idx in origin_indices], dtype=np.float64)
assert y_val_true.shape == (num_val_origins, 7)

feature_data = {}
expected_counts = {0: 36, 1: 53, 2: 65, 3: 70}

for lvl in range(4):
    cfg = FeatureConfig.from_level(lvl)
    fb = FeatureBuilder(cfg).fit(train_full)
    
    # Train features (giữ nguyên độ dài để áp dụng mask 6,293 mẫu)
    df_feat_train = fb.transform(train_full, drop_warmup=False)
    feat_cols = [c for c in df_feat_train.columns if c not in ['Ngày', 'Lượng mưa']]
    assert len(feat_cols) == expected_counts[lvl]
    
    # Common training origins mask: loại 29 origins đầu -> đúng 6,293 mẫu
    mask = np.ones(len(train_full) - 7, dtype=bool)
    mask[:29] = False
    
    X_train, Y_train, _ = build_purged_direct_samples(
        df_feat_train, feature_cols=feat_cols, target_col='Lượng mưa', horizon=7, origin_mask=mask
    )
    assert len(X_train) == 6293
    
    # Validation features qua Mode B
    fset_val = fb.build_feature_set(val_set, context_df=train_full)
    X_val = fset_val.X.iloc[:num_val_origins].values.astype(np.float64)
    assert X_val.shape == (num_val_origins, expected_counts[lvl])
    
    feature_data[f"F{lvl}"] = {
        "feature_cols": feat_cols,
        "feature_count": expected_counts[lvl],
        "X_train": X_train,
        "Y_train": Y_train,
        "X_val": X_val,
    }
    print(f"[F{lvl}] Features: {expected_counts[lvl]} cols | X_train: {X_train.shape} | X_val: {X_val.shape}")


## 3. Khởi tạo & Cấu hình 16 Kịch bản Hồi quy Đa bước Trực tiếp (SC-12 đến SC-27)

In [3]:
def create_model_instance(model_family: str):
    if model_family == "lr_direct":
        return MultiOutputRegressor(LinearRegression())
    elif model_family == "rf_direct":
        return RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    elif model_family == "xgboost_direct":
        return MultiOutputRegressor(
            XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)
        )
    elif model_family == "lightgbm_direct":
        return MultiOutputRegressor(
            LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, verbosity=-1, random_state=42, n_jobs=-1)
        )
    raise ValueError(f"Unknown family: {model_family}")

scenario_specs = [
    # Linear Regression (SC-12..15)
    {"scenario_id": "SC-12", "model_family": "lr_direct", "feature_set": "F0", "param_count": 36 * 7},
    {"scenario_id": "SC-13", "model_family": "lr_direct", "feature_set": "F1", "param_count": 53 * 7},
    {"scenario_id": "SC-14", "model_family": "lr_direct", "feature_set": "F2", "param_count": 65 * 7},
    {"scenario_id": "SC-15", "model_family": "lr_direct", "feature_set": "F3", "param_count": 70 * 7},
    # Random Forest (SC-16..19)
    {"scenario_id": "SC-16", "model_family": "rf_direct", "feature_set": "F0", "param_count": 100 * 1024},
    {"scenario_id": "SC-17", "model_family": "rf_direct", "feature_set": "F1", "param_count": 100 * 1024},
    {"scenario_id": "SC-18", "model_family": "rf_direct", "feature_set": "F2", "param_count": 100 * 1024},
    {"scenario_id": "SC-19", "model_family": "rf_direct", "feature_set": "F3", "param_count": 100 * 1024},
    # XGBoost Direct (SC-20..23)
    {"scenario_id": "SC-20", "model_family": "xgboost_direct", "feature_set": "F0", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-21", "model_family": "xgboost_direct", "feature_set": "F1", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-22", "model_family": "xgboost_direct", "feature_set": "F2", "param_count": 7 * 100 * 32},
    {"scenario_id": "SC-23", "model_family": "xgboost_direct", "feature_set": "F3", "param_count": 7 * 100 * 32},
    # LightGBM Direct (SC-24..27)
    {"scenario_id": "SC-24", "model_family": "lightgbm_direct", "feature_set": "F0", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-25", "model_family": "lightgbm_direct", "feature_set": "F1", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-26", "model_family": "lightgbm_direct", "feature_set": "F2", "param_count": 7 * 100 * 31},
    {"scenario_id": "SC-27", "model_family": "lightgbm_direct", "feature_set": "F3", "param_count": 7 * 100 * 31},
]

print(f"Đã khởi tạo danh sách {len(scenario_specs)} kịch bản học máy dạng bảng trực tiếp.")


## 4. Huấn luyện Single-Fit trên Dev Train & Đánh giá Rolling Origins trên Validation

In [4]:
forecasts_dict = {}
benchmark_rows = []
horizon_rows = []

for spec in scenario_specs:
    sc_id = spec["scenario_id"]
    m_fam = spec["model_family"]
    f_set = spec["feature_set"]
    model_name = f"{m_fam}_{f_set}"
    print(f">>> [{sc_id}] Đang huấn luyện & suy luận {model_name}...")

    X_tr = feature_data[f_set]["X_train"]
    Y_tr = feature_data[f_set]["Y_train"]
    X_v = feature_data[f_set]["X_val"]

    model = create_model_instance(m_fam)

    # Fit trên 6,293 mẫu Dev Train
    t0 = time.time()
    model.fit(X_tr, Y_tr)
    t_fit = time.time() - t0

    # Suy luận trên 724 Validation origins (Clipped non-negative)
    t0 = time.time()
    preds = model.predict(X_v)
    preds = np.clip(preds, 0.0, None)
    t_infer = time.time() - t0

    forecasts_dict[f"{sc_id}_{model_name}"] = preds
    print(f"    Fit: {t_fit:.2f}s | Infer: {t_infer:.2f}s ({(t_infer/num_val_origins)*1000:.2f} ms/origin)")

    # Metrics chuẩn hóa
    row_df = compute_benchmark_metrics(
        y_true=y_val_true,
        y_pred=preds,
        model_name=model_name,
        group="Machine Learning",
        strategy=f"Direct ML ({f_set})",
        training_time=t_fit,
        inference_time=t_infer,
        parameter_count=spec["param_count"],
        status=ExecutionStatus.SUCCESS,
        rain_threshold=1.0,
        scale=CANONICAL_DEV_TRAIN_MASE_SCALE,
        horizon=7,
    )
    row_df.insert(0, "scenario_id", sc_id)
    row_df.insert(4, "feature_set", f_set)
    row_df.insert(5, "feature_count", feature_data[f_set]["feature_count"])
    benchmark_rows.append(row_df)

    # Metrics theo từng bước horizon h=1..7
    detailed = evaluate_point_forecast_metrics(y_true=y_val_true, y_pred=preds, scale=CANONICAL_DEV_TRAIN_MASE_SCALE, horizon=7)
    h_rec = {"scenario_id": sc_id, "model": model_name, "feature_set": f_set, "MAE": detailed["MAE"], "RMSE": detailed["RMSE"], "MASE": detailed["MASE"]}
    for h in range(1, 8):
        h_rec[f"MAE@h{h}"] = detailed[f"mae_h{h}"]
        h_rec[f"RMSE@h{h}"] = detailed[f"rmse_h{h}"]
        h_rec[f"MASE@h{h}"] = detailed[f"mase_lag1_h{h}"]
    horizon_rows.append(h_rec)

print("\nHoàn thành 16 kịch bản Direct Machine Learning.")


## 5. Phân tích Benchmark A & Bảng Xếp hạng Validation Tổng thể

In [5]:
df_results = pd.concat(benchmark_rows, ignore_index=True).sort_values(by="MAE", ascending=True).reset_index(drop=True)
df_horizons = pd.DataFrame(horizon_rows)

# Lưu kết quả vào data/processed/results
results_dir = ROOT_DIR / "data" / "processed" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

val_csv = results_dir / "ml_direct_val_results.csv"
val_npz = results_dir / "ml_direct_val_forecasts.npz"
h_csv = results_dir / "ml_direct_val_horizon_breakdown.csv"

df_results.to_csv(val_csv, index=False)
df_horizons.to_csv(h_csv, index=False)

payload = {"y_true": y_val_true, "origin_dates": origin_dates}
payload.update(forecasts_dict)
np.savez_compressed(val_npz, **payload)

# Tương thích file cũ
df_results.to_csv(results_dir / "ml_direct_results.csv", index=False)
np.savez_compressed(results_dir / "ml_direct_forecasts.npz", **payload)

print("=== KẾT QUẢ BENCHMARK A: KHẢO SÁT ABLATION TRÊN LIGHTGBM DIRECT ===")
lgb_ablation = df_results[df_results['model'].str.startswith('lightgbm_direct')].sort_values(by='MAE')
display(lgb_ablation[['scenario_id', 'model', 'feature_set', 'feature_count', 'MAE', 'MSE', 'RMSE', 'MASE']])
print(f">> Tập đặc trưng tối ưu cho LightGBM Direct: {lgb_ablation.iloc[0]['feature_set']} (MAE = {lgb_ablation.iloc[0]['MAE']:.4f} mm)\n")

print("=== BẢNG XẾP HẠNG TOÀN DIỆN 16 KỊCH BẢN ML DIRECT (Ranked by MAE) ===")
disp_cols = ['scenario_id', 'model', 'feature_set', 'MAE', 'MSE', 'RMSE', 'MASE', 'training_time', 'inference_time', 'status']
display(df_results[disp_cols])

print("\n=== CHI TIẾT THEO HORIZON (MAE tại h=1..7) ===")
display(df_horizons[['scenario_id', 'model', 'feature_set', 'MAE'] + [f'MAE@h{h}' for h in range(1, 8)]])
